# C. Multi Query 검색과 평가

AI 기본법 QA 프로젝트에서 C 담당 단계(Multi Query, 평가)를 정리한 노트북이다.

| 단계 | 내용 | 코드 | API 호출 |
|---|---|---|---|
| 1 | 설정 확인 | `common/`, `retrieval/dense.py`(B) | 없음 |
| 2 | RRF 병합 원리 | `retrieval/multi_query.py` `rrf_merge` | 없음 |
| 3 | 평가 결과 조회 | `evaluation/results_view.py` | 없음 (저장된 결과) |
| 4 | 문항 하나 + 청크 원문 | `evaluation/results_view.py` `get_question` | 없음 (Qdrant만) |
| 5 | 리포트 생성 | `evaluation/report.py` | 없음 |
| 6 | (선택) 질의 확장·질문 추적 | `retrieval/multi_query.py`, `evaluation/trace.py` | ⚠️ `RUN_LIVE = True`일 때만 |

> **API 호출 절감**: MonoRouter는 분당 30회 제한이고 키를 여러 명이 함께 쓴다.
> 이 노트북은 기본적으로 **저장된 평가 결과만 읽는다** (`RUN_LIVE = False`). 평가 실행은 터미널에서 한 번만 한다.
>
> 실제 데이터: A가 공유한 `data_ingestion_law21311_v1`(조 단위 46개 청크, 스냅샷 복구). `config.data_version`이 `dummy-v1`인 결과는 더미 데이터(17개 청크) 기준이라 성능으로 보고하지 않는다.

## 1. 설정 확인

In [1]:
RUN_LIVE = False  # True로 바꾸면 6장에서 MonoRouter를 호출한다 (질의 확장 1회, 질문 추적 수 회)

COLLECTION = "data_ingestion_law21311_v1"  # A가 공유한 실제 데이터 (더미 평가는 "tmp_c_law_articles")
STAMP = None                       # 볼 평가 실행 시각, None이면 가장 최근

from common.config import MODEL, EMBEDDING_MODEL
from common.qdrant import get_qdrant_client

print("LLM:", MODEL, "/ 임베딩:", EMBEDDING_MODEL)
print("컬렉션:", COLLECTION, "청크 수:", get_qdrant_client().count(COLLECTION).count)

LLM: gpt-5.4-mini / 임베딩: text-embedding-3-small
컬렉션: data_ingestion_law21311_v1 청크 수: 46


평가 실행은 터미널에서 한다. 기본값은 **실제 골든셋 10문항**(`evaluation/golden_set.jsonl`)과 **A의 컬렉션**(`data_ingestion_law21311_v1`)이다.
호출은 묶어서 보내고 결과는 캐시(`evaluation/.cache`)에 저장되어, 같은 평가를 다시 실행하면 호출이 0회다.

```bash
# A의 컬렉션 점검 (API 0회)
uv run python -m evaluation.check_collection

# 개발 중: 검색만 (첫 실행 = 질의 확장 1회 + 임베딩 1회)
uv run python -m evaluation.run_eval --k 5

# 최종: 답변은 baseline·multi_query만, 채점은 규칙 기반(API 0회)
uv run python -m evaluation.run_eval --k 5 --answer temp      # B 답변 함수 연결 후 --answer shared
```

## 2. RRF 병합 원리 (API 호출 없음)

질의마다 Dense 결과 순위가 다르다. RRF는 점수 대신 **순위**만 사용해 `1 / (60 + 순위)`를 더한다.
여러 질의에서 꾸준히 상위에 나온 청크가 위로 올라오고, `chunk_id` 기준으로 중복이 제거된다.
Dense 유사도와 RRF 점수는 의미가 달라서 섞지 않는다.

In [2]:
from retrieval.multi_query import rrf_merge

def toy(chunk_id):
    return {"chunk_id": chunk_id, "article": chunk_id, "content": ""}

ranked_lists = [
    [toy("제1조"), toy("제2조"), toy("제3조")],  # 원질문
    [toy("제2조"), toy("제4조")],               # 확장1
    [toy("제2조"), toy("제1조")],               # 확장2
]
for r in rrf_merge(ranked_lists, top_k=4):
    print(r["rank"], r["chunk_id"], round(r["retrieval_score"], 4))

1 제2조 0.0489
2 제1조 0.0325
3 제4조 0.0161
4 제3조 0.0159


## 3. 평가 결과 조회 (저장된 결과)

- 검색: Hit@K, Recall@K, MRR, 유형별(기준/MQ/RR/둘다), Rerank 전 후보 Recall
- 답변: 정답 근거 사용률, 오거절률, 답변 불가 거절률, 규칙 채점(필수 요소 포함률, 근거 겹침), (선택) LLM 채점
- Multi Query: 확장 질의별 정답 순위, 원질문 결과 대비 새로 들어온/밀려난 청크
- Rerank가 연결되지 않았으면 "⏸ 아직 연결되지 않았습니다"로 표시된다

In [3]:
from evaluation.results_view import get_results, list_runs, print_results

print("실제 골든셋 실행:", [r["stamp"] for r in list_runs("golden_set.jsonl") if r["compatible"]][:5])
try:
    data = get_results(STAMP)  # STAMP=None → 실제 골든셋 실행 중 최신
    STAMP = data["stamp"]
    print_results(data, detail=True)
except FileNotFoundError as e:
    data = None
    print(e)

실제 골든셋 실행: []
실제 골든셋(golden_set.jsonl) 평가 결과가 없습니다. evaluation.run_eval 을 먼저 실행하세요. (더미 결과는 stamp를 지정해서 조회)


## 4. 문항 하나 + 청크별 역할과 원문 (Qdrant에서 원문 조회, MonoRouter 호출 없음)

In [4]:
from evaluation.results_view import get_question

if data:
    question_id = data["questions"][0]["id"]
    q = get_question(STAMP, question_id, with_source=True)
    print(f"[{q['id']}] {q['question']}  | 정답: {q['gold']}")
    for c in q["chunks"]:
        mark = {True: "정답", False: "", None: "?"}[c["is_gold"]]
        print(f"- {c['label']} {mark} {c['modes']}")
        print("    ", (c["content"] or "(원본에서 찾지 못함)")[:100])
else:
    print("평가 결과가 없어 건너뜁니다.")

평가 결과가 없어 건너뜁니다.


## 5. 마크다운 리포트 생성 (API 호출 없음)

In [5]:
from evaluation.report import build_report
from evaluation.results_view import RESULT_DIR, load_runs

if data:
    report_path = RESULT_DIR / f"{STAMP}_report.md"
    report_path.write_text(build_report(STAMP, load_runs(STAMP)), encoding="utf-8")
    print(report_path)
else:
    print("평가 결과가 없어 건너뜁니다.")

평가 결과가 없어 건너뜁니다.


## 6. (선택) 질의 확장·질문 추적 ⚠️ API 호출

`RUN_LIVE = True`일 때만 실행된다. 골든셋에 없는 질문 하나를 모드별로 따라가 본다.
호출 수: 질의 확장 1회 + 임베딩 2회(원질문, 확장 질의 묶음) + 답변 2회(baseline, multi_query) 정도.
같은 질문을 다시 실행하면 캐시로 0회다.

In [6]:
if RUN_LIVE:
    from IPython.display import Markdown, display
    from evaluation.trace import trace_question, trace_to_markdown

    trace = trace_question(
        "AI가 내 대출을 거절했는데 왜 그런 결정이 나왔는지 설명을 들을 수 있나요?",  # 골든셋 2번
        modes=["baseline", "rerank", "multi_query"],
        final_k=5,
        collection=COLLECTION,
        answer="temp",
        gold_articles=["제3조", "제34조"],
    )
    display(Markdown(trace_to_markdown(trace)))
else:
    print("RUN_LIVE = False: API를 호출하지 않습니다. 저장된 추적 결과는 evaluation/results/trace_*.md")

RUN_LIVE = False: API를 호출하지 않습니다. 저장된 추적 결과는 evaluation/results/trace_*.md


## 7. 실제 데이터로 전환할 때

| 바꿀 것 | 방법 |
|---|---|
| 데이터 | A의 스냅샷을 복구한 `data_ingestion_law21311_v1` (`--collection` 생략 시 기본값, `QDRANT_COLLECTION`으로 변경) |
| 골든셋 | `evaluation/golden_set.jsonl` (조 번호 기준 판정) |
| Rerank | B의 `retrieval/rerank.py` `rerank(question, candidates, final_k)` 추가 → 자동으로 rerank·combined 실행 |
| 답변 | B의 `generation/answer.py` `generate_answer` → `--answer shared` |

```bash
uv run python -m evaluation.run_eval --k 5 --answer shared --data-version <A의 버전>
```

이 노트북은 그 결과를 3~5장에서 그대로 읽는다.